# EMHA: Colab bootstrap (smoke test)


Bootstrap only: this notebook clones the repo, installs, sets paths, and calls
modules (CLAUDE.md). Pipeline logic, model code, and every hyperparameter live
in `src/` (`src/utils/config.py`); never add them here. If `pip` upgrades torch,
restart the runtime and run the cells again from the top.

Needs the Colab secret `GITHUB_TOKEN` with notebook access switched on.

In [ ]:
# Notebook parameters
SUBSET = "pilot"  # "pilot" or None (all participants)
ANALYSIS = "primary"  # "primary" | "full"
DEVICE = "cuda"  # "cuda" | "cpu"
REPO = "Yeeejj/EMHA"
REPO_DIR = "/content/EMHA"
DRIVE_DATA_ROOT = "/content/drive/MyDrive/EMHA/DATASET"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/EMHA/outputs"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

# Full clone with all history and tags (never shallow): the protocol-frozen tag
# check needs them. The token goes in a one-off HTTP header, so it is never
# written to .git/config, a URL, or the output.
import base64
import os
import subprocess

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; using it as is")
else:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
         "clone", f"https://github.com/{REPO}.git", REPO_DIR],
        check=True,
    )
    del auth
del token
os.chdir(REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# Record exactly what ran: commit, protocol tag, torch / CUDA.
!git rev-parse HEAD
!git describe --tags --always
!git tag --list protocol-frozen
import torch

print("torch", torch.__version__, "| CUDA build", torch.version.cuda,
      "| CUDA available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
os.makedirs(DRIVE_OUTPUT_ROOT, exist_ok=True)
os.environ["EMHA_DATA_ROOT"] = DRIVE_DATA_ROOT
os.environ["EMHA_OUTPUT_ROOT"] = DRIVE_OUTPUT_ROOT
print({k: os.environ[k] for k in ("EMHA_DATA_ROOT", "EMHA_OUTPUT_ROOT")})
print({"SUBSET": SUBSET, "ANALYSIS": ANALYSIS, "DEVICE": DEVICE})

The smoke test runs the whole pipeline on synthetic data in a temporary folder; it never writes to Drive's DATASET.

In [ ]:
!python smoke_test.py
if _exit_code != 0:
    raise SystemExit(f"smoke_test.py failed (exit {_exit_code})")